# Eksperimen Machine Learning - California Housing Dataset
**Nama:** Athalie Aurora

Notebook ini mengikuti Template Eksperimen MSML yang mencakup:
1. Data Loading
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Kesimpulan

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

print('Libraries imported successfully!')

## 2. Data Loading

In [ ]:
# Load dataset
california = fetch_california_housing(as_frame=True)
df = california.frame

# Simpan raw data
os.makedirs('../california_housing_raw', exist_ok=True)
df.to_csv('../california_housing_raw/california_housing_raw.csv', index=False)

print(f'Dataset shape: {df.shape}')
print(f'\nFeatures: {california.feature_names}')
print(f'Target: MedHouseVal (Median House Value)')
df.head()

In [ ]:
# Info dataset
df.info()

In [ ]:
# Statistik deskriptif
df.describe()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Cek missing values
print('Missing Values per Column:')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# Cek duplikat
print(f'Jumlah duplikat: {df.duplicated().sum()}')

In [ ]:
# Distribusi target variable
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(df['MedHouseVal'], bins=50, color='steelblue', edgecolor='black')
plt.title('Distribusi MedHouseVal')
plt.xlabel('Median House Value')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
sns.boxplot(y=df['MedHouseVal'], color='steelblue')
plt.title('Boxplot MedHouseVal')

plt.tight_layout()
plt.show()

In [ ]:
# Distribusi semua fitur
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=50, color='steelblue', edgecolor='black')
    axes[i].set_title(f'Distribusi {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, linewidths=0.5)
plt.title('Correlation Matrix - California Housing')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot fitur vs target
features = california.feature_names
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(features):
    axes[i].scatter(df[feat], df['MedHouseVal'], alpha=0.1, color='steelblue')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('MedHouseVal')
    axes[i].set_title(f'{feat} vs MedHouseVal')

plt.tight_layout()
plt.show()

In [ ]:
# Deteksi outlier dengan IQR
print('Deteksi Outlier (IQR Method):')
for col in df.columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    print(f'  {col}: {outliers} outliers ({outliers/len(df)*100:.2f}%)')

## 4. Data Preprocessing

In [ ]:
# Pisahkan fitur dan target
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

print(f'Shape X: {X.shape}')
print(f'Shape y: {y.shape}')

In [ ]:
# Handle missing values (imputation)
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)
X_imputed = pd.DataFrame(X_imputed, columns=X.columns)

print('Missing values setelah imputation:')
print(X_imputed.isnull().sum())

In [ ]:
# Handle outlier dengan clipping (IQR)
def clip_outliers(df_input):
    df_clipped = df_input.copy()
    for col in df_clipped.columns:
        Q1 = df_clipped[col].quantile(0.25)
        Q3 = df_clipped[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df_clipped[col] = df_clipped[col].clip(lower, upper)
    return df_clipped

X_clipped = clip_outliers(X_imputed)
print('Outlier handling selesai.')
X_clipped.describe()

In [ ]:
# Feature Engineering: tambah fitur baru
X_clipped['RoomsPerHousehold'] = X_clipped['AveRooms'] / X_clipped['HouseAge'].replace(0, 1)
X_clipped['BedroomsPerRoom'] = X_clipped['AveBedrms'] / X_clipped['AveRooms'].replace(0, 1)
X_clipped['PopulationPerHousehold'] = X_clipped['Population'] / X_clipped['AveOccup'].replace(0, 1)

print(f'Shape setelah feature engineering: {X_clipped.shape}')
X_clipped.head()

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_clipped, y, test_size=0.2, random_state=42
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

In [ ]:
# Feature Scaling (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_clipped.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X_clipped.columns)

print('Scaling selesai.')
X_train_scaled.describe()

In [ ]:
# Simpan hasil preprocessing
os.makedirs('california_housing_preprocessing', exist_ok=True)

train_df = X_train_scaled.copy()
train_df['MedHouseVal'] = y_train.values

test_df = X_test_scaled.copy()
test_df['MedHouseVal'] = y_test.values

train_df.to_csv('california_housing_preprocessing/train.csv', index=False)
test_df.to_csv('california_housing_preprocessing/test.csv', index=False)

print(f'Train set saved: {train_df.shape}')
print(f'Test set saved:  {test_df.shape}')

## 5. Kesimpulan

- Dataset California Housing memiliki **20.640 baris** dan **8 fitur** numerik.
- **Tidak ada missing values** pada dataset ini.
- Terdapat outlier pada beberapa fitur seperti `AveRooms`, `AveBedrms`, dan `Population` yang sudah ditangani dengan metode IQR clipping.
- Ditambahkan **3 fitur baru** hasil feature engineering: `RoomsPerHousehold`, `BedroomsPerRoom`, `PopulationPerHousehold`.
- Data dibagi menjadi **80% train** dan **20% test**.
- Fitur dinormalisasi menggunakan **StandardScaler**.
- Dataset siap digunakan untuk pelatihan model regresi.